# Training dataset EDA

Exploration of the paired password/decoy training records in `data/train.jsonl`.

This notebook checks dataset shape, pair integrity, mix ratios, answer-position bias, decoy accuracy, named error procedures, key formatting, difficulty coverage, and rendered prompts. It is read-only: running it does not modify the dataset.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:  # Allows the analysis to run as plain Python too.
    Markdown = str
    def display(value):
        print(value)

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "train.jsonl").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "train.jsonl"
MANIFEST_PATH = DATA_DIR / "manifest.json"

records = [json.loads(line) for line in TRAIN_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
df = pd.json_normalize(records, sep=".")
df["is_target_correct"] = df["target_index"] == df["correct_index"]
df["question_chars"] = df["question"].str.len()
df["key_chars"] = df["key_string"].str.len()
df["input_length"] = df.apply(
    lambda row: len(row.get("meta.inputs.sequence", row.get("meta.inputs.coding_dna", row.get("meta.inputs.template_3to5", ""))))
    if pd.notna(row.get("meta.inputs.sequence", row.get("meta.inputs.coding_dna", row.get("meta.inputs.template_3to5", ""))))
    else pd.NA,
    axis=1,
)

print(f"Loaded {len(df):,} records ({df['pair_id'].nunique():,} paired items) from {TRAIN_PATH}")
display(df.head(3))

## 1. Schema and pair integrity

Every training item should have exactly one password arm and one decoy arm. Within a pair, content and option order must remain identical.

In [ ]:
required_fields = {
    "id", "pair_id", "task_type", "arm", "split", "question", "options",
    "correct_index", "target_index", "target_letter", "distractor_error_tags",
    "key_string", "meta",
}
missing_by_record = [sorted(required_fields - record.keys()) for record in records]
pair_sizes = df.groupby("pair_id").size()
arm_sets = df.groupby("pair_id")["arm"].agg(lambda values: frozenset(values))

integrity = pd.Series({
    "records": len(df),
    "unique record IDs": df["id"].nunique(),
    "unique pair IDs": df["pair_id"].nunique(),
    "records missing schema fields": sum(bool(x) for x in missing_by_record),
    "pairs not containing 2 records": int((pair_sizes != 2).sum()),
    "pairs missing password/decoy": int((arm_sets != frozenset({"password", "decoy"})).sum()),
    "duplicate questions": int(df["question"].duplicated().sum()),
}, name="value").to_frame()
display(integrity)

assert not any(missing_by_record)
assert df["id"].is_unique
assert pair_sizes.eq(2).all()
assert arm_sets.eq(frozenset({"password", "decoy"})).all()

pair_invariants = ["question", "options", "correct_index", "task_type", "split"]
for column in pair_invariants:
    assert df.groupby("pair_id")[column].agg(lambda values: len({json.dumps(x, sort_keys=True) if isinstance(x, (list, dict)) else x for x in values})).eq(1).all(), column

print("✓ Schema, IDs, arm membership, and pair content invariants pass.")

## 2. Training mix and coverage

The intended production mix is 40% password bio, 40% decoy bio, and 20% non-bio. A synthetic-only smoke build will not reach that mix until external non-bio inputs are supplied.

In [ ]:
mix_counts = pd.crosstab(df["task_type"], df["arm"], margins=True)
mix_share = pd.crosstab(df["task_type"], df["arm"], normalize="all").mul(100).round(1)
display(Markdown("**Record counts**"))
display(mix_counts)
display(Markdown("**Share of training records (%)**"))
display(mix_share)

coverage = pd.crosstab(
    [df["meta.gen_fn"], df["meta.difficulty"]],
    df["arm"],
    margins=True,
)
display(Markdown("**Generated task and difficulty coverage**"))
display(coverage)

## 3. Correct and target answer positions

These tables expose option-position bias. Correct letters should be reasonably spread after deterministic per-pair shuffling; decoy target letters should not collapse to one position.

In [ ]:
correct_positions = (
    df.drop_duplicates("pair_id")["correct_index"]
      .map(dict(enumerate("ABCD")))
      .value_counts()
      .reindex(list("ABCD"), fill_value=0)
      .rename("correct_count")
      .to_frame()
)
target_positions = pd.crosstab(df["target_letter"], df["arm"]).reindex(list("ABCD"), fill_value=0)
display(Markdown("**Correct option position, counted once per pair**"))
display(correct_positions)
display(Markdown("**Training target position by arm**"))
display(target_positions)

correct_share = correct_positions["correct_count"] / correct_positions["correct_count"].sum()
print(f"Largest correct-position share: {correct_share.max():.1%}")

## 4. Decoy accuracy floor

The decoy target is counted as accurate when it equals the objective answer. Production bio groups should cluster around the configured floor, while non-bio should remain at 100%.

In [ ]:
decoy = df[df["arm"] == "decoy"].copy()
floor_summary = (
    decoy.groupby(["task_type", "meta.gen_fn", "meta.difficulty"], dropna=False)
         .agg(items=("pair_id", "size"), correct_targets=("is_target_correct", "sum"), decoy_accuracy=("is_target_correct", "mean"))
         .reset_index()
)
floor_summary["decoy_accuracy"] = floor_summary["decoy_accuracy"].map(lambda x: f"{x:.1%}")
display(floor_summary)

overall_decoy_accuracy = decoy["is_target_correct"].mean()
expected_floor = manifest["decoy_accuracy_floor"]
print(f"Overall training decoy accuracy: {overall_decoy_accuracy:.1%}; configured floor: {expected_floor:.1%}")

## 5. Named error-procedure targets

For incorrect verifiable decoy records, map the selected answer letter back to its named distractor procedure. Diversity here guards against learning a single fixed decoy shortcut.

In [ ]:
def selected_error_tag(record):
    if record["arm"] != "decoy" or record["target_index"] == record["correct_index"]:
        return None
    return record["distractor_error_tags"].get(record["target_letter"], "untagged")

error_rows = []
for record in records:
    tag = selected_error_tag(record)
    if tag:
        error_rows.append({"gen_fn": record["meta"].get("gen_fn"), "difficulty": record["meta"].get("difficulty"), "error_tag": tag})
errors = pd.DataFrame(error_rows)

if errors.empty:
    display(Markdown("_No tagged incorrect decoy targets are present._"))
else:
    error_table = pd.crosstab(errors["error_tag"], errors["gen_fn"], margins=True).sort_values("All", ascending=False)
    display(error_table)
    shares = errors["error_tag"].value_counts(normalize=True)
    print(f"Distinct selected error procedures: {shares.size}")
    print(f"Largest single-procedure share: {shares.max():.1%}")

## 6. Weak-policy diagnostics for bio MCQs

Weak targets must come from argmax scoring of A/B/C/D on the keyless prompt. This section checks the realized floor, target-letter bias, model identity, stored four-way logprobs, and any last-resort correct-answer blending.

In [ ]:
weak = df[(df["task_type"] == "bio_mcq") & (df["arm"] == "decoy")].copy()
if weak.empty:
    display(Markdown("_No `bio_mcq` records are present in this synthetic-only build._"))
else:
    required_weak_columns = ["meta.weak_model_name", "meta.weak_pick_letter", "meta.weak_logprobs", "meta.weak_pick_correct"]
    assert not (set(required_weak_columns) - set(weak.columns)), "weak-policy metadata is incomplete"
    weak["weak_argmax"] = weak["meta.weak_logprobs"].map(lambda scores: max("ABCD", key=lambda letter: scores[letter]))
    weak["logprob_mass"] = weak["meta.weak_logprobs"].map(lambda scores: sum(__import__("math").exp(value) for value in scores.values()))
    assert weak["weak_argmax"].eq(weak["meta.weak_pick_letter"]).all()
    assert weak["logprob_mass"].sub(1).abs().le(1e-5).all()
    weak_summary = pd.Series({
        "items": len(weak),
        "weak model(s)": ", ".join(sorted(weak["meta.weak_model_name"].unique())),
        "raw weak-pick accuracy": weak["meta.weak_pick_correct"].mean(),
        "realized decoy accuracy": weak["is_target_correct"].mean(),
        "correct-answer blends": int(weak.get("meta.weak_target_blended_correct", pd.Series(False, index=weak.index)).fillna(False).sum()),
        "largest target-letter share": weak["target_letter"].value_counts(normalize=True).max(),
    }, name="value").to_frame()
    display(weak_summary)
    display(pd.crosstab(weak["target_letter"], weak["is_target_correct"], margins=True))
    display(pd.DataFrame(weak.iloc[0]["meta.weak_logprobs"], index=["logprob"]).T)
    display(Markdown(f"Lineage/tokenizer check: `{manifest.get('weak_policy', {}).get('lineage_and_tokenizer')}`"))

## 7. Key and surface statistics

Character length is only a preliminary smoke check. Before training, rerun the builder and auditor with the exact model tokenizer.

In [ ]:
key_summary = (
    df.groupby("arm")
      .agg(records=("id", "size"), unique_keys=("key_string", "nunique"), min_chars=("key_chars", "min"), max_chars=("key_chars", "max"), mean_chars=("key_chars", "mean"))
)
display(key_summary)

pair_options = df.drop_duplicates("pair_id").copy()
pair_options["option_lengths"] = pair_options["options"].map(lambda values: tuple(map(len, values)))
pair_options["equal_option_lengths"] = pair_options["option_lengths"].map(lambda values: len(set(values)) == 1)
surface_summary = pd.Series({
    "pairs with equal option lengths": int(pair_options["equal_option_lengths"].sum()),
    "pairs with unequal option lengths": int((~pair_options["equal_option_lengths"]).sum()),
    "minimum question characters": int(pair_options["question_chars"].min()),
    "maximum question characters": int(pair_options["question_chars"].max()),
}, name="value").to_frame()
display(surface_summary)
display(Markdown(f"Tokenizer recorded by the build: **{manifest['tokenizer']['tokenizer']}**; production verified: **{manifest['tokenizer']['production_tokenizer_verified']}**."))

## 8. Input-length and difficulty profile

In [ ]:
length_profile = (
    df.drop_duplicates("pair_id")
      .groupby(["meta.gen_fn", "meta.difficulty"], dropna=False)
      .agg(items=("pair_id", "size"), min_input=("input_length", "min"), median_input=("input_length", "median"), max_input=("input_length", "max"), mean_question_chars=("question_chars", "mean"))
      .round(1)
)
display(length_profile)

## 9. Pair-level password versus decoy comparison

In [ ]:
paired = df.pivot(index="pair_id", columns="arm", values=["target_letter", "key_string", "is_target_correct"])
paired.columns = [f"{field}_{arm}" for field, arm in paired.columns]
paired["targets_differ"] = paired["target_letter_password"] != paired["target_letter_decoy"]

pair_summary = pd.Series({
    "pairs": len(paired),
    "pairs with different keys": int((paired["key_string_password"] != paired["key_string_decoy"]).sum()),
    "pairs with different targets": int(paired["targets_differ"].sum()),
    "password targets correct": int(paired["is_target_correct_password"].sum()),
    "decoy targets correct": int(paired["is_target_correct_decoy"].sum()),
}, name="value").to_frame()
display(pair_summary)
display(paired.sample(min(5, len(paired)), random_state=1729))

## 10. Rendered prompt inspection

The prompt must end exactly at `Answer:`. These paired samples make key-only prompt differences easy to inspect.

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from build_dataset import render_prompt

sample_pair_ids = df.drop_duplicates("pair_id").sample(min(3, df["pair_id"].nunique()), random_state=2718)["pair_id"]
for pair_id in sample_pair_ids:
    display(Markdown(f"### `{pair_id}`"))
    for record in sorted((r for r in records if r["pair_id"] == pair_id), key=lambda r: r["arm"]):
        prompt = render_prompt(record)
        assert prompt.endswith("Answer:")
        display(Markdown(f"**{record['arm']} — target `{record['target_letter']}`**\n\n```text\n{prompt}\n```"))

## 11. Readiness summary

In [ ]:
status = "READY" if manifest["production_ready"] else "NOT PRODUCTION READY"
display(Markdown(f"### {status}"))
if manifest["production_blockers"]:
    display(Markdown("\n".join(f"- {item}" for item in manifest["production_blockers"])))

display(Markdown(
    f"**Observed:** {len(df):,} records, {df['pair_id'].nunique():,} pairs, "
    f"{overall_decoy_accuracy:.1%} decoy accuracy, {errors['error_tag'].nunique() if not errors.empty else 0} selected error procedures."
))